# SWOT Tidal Residual Analysis — Canal Baker

Loads SWOT WSE parquet from S3, computes FES2022 and SHOA+offset tidal
predictions, and characterizes tidal signal across the Canal Baker network.

**Prerequisites:**
- `bahia_orange_predictions.csv` in project root (manually exported from SHOA predictor)
- FES2022 model files (downloaded automatically to `/scratch/fes2022/` on first run)

**Input:** `s3://esip-qhub/canal-baker-tides/swot_wse_samples.parquet`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import s3fs
import pyTMD
from pathlib import Path

INPUT_PATH = "s3://esip-qhub/canal-baker-tides/swot_wse_samples.parquet"
FES2022_DIR = Path("/scratch/fes2022")
SHOA_CSV = Path("bahia_orange_predictions.csv")

# Time offsets (minutes) from Pub. 3009 — tide arrives this many minutes
# after Bahía Orange at each SHOA anchor port. Look these up in Pub. 3009
# for Puerto Francisco (P07) and Puerto Brown (P12) and update below.
SHOA_OFFSETS = {
    "P07": 0,   # TODO: replace with published Pub. 3009 value for Puerto Francisco
    "P12": 0,   # TODO: replace with published Pub. 3009 value for Puerto Brown
}

In [ ]:
fs = s3fs.S3FileSystem()
with fs.open(INPUT_PATH, "rb") as f:
    df = pd.read_parquet(f)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

points = pd.read_csv("sample_points.csv")

print(f"Loaded {len(df)} observations across {df['point_id'].nunique()} points")
print(f"Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")
display(df.head())

In [ ]:
FES2022_DIR.mkdir(parents=True, exist_ok=True)

existing = list(FES2022_DIR.glob("**/*.nc"))
if not existing:
    print("Downloading FES2022 from AVISO — requires AVISO credentials in ~/.netrc")
    print("See: https://pytmd.readthedocs.io/en/latest/getting_started/FES.html")
    import subprocess
    r = subprocess.run(
        ["python", "-m", "pyTMD.download_FES_Tidal_Models",
         "--directory", str(FES2022_DIR),
         "--model", "FES2022",
         "--tide", "ocean"],
        capture_output=True, text=True
    )
    print(r.stdout[-2000:] if r.stdout else "")
    if r.returncode != 0:
        print("DOWNLOAD ERROR:", r.stderr[-1000:])
        raise RuntimeError("FES2022 download failed — see output above and pyTMD docs")
else:
    print(f"FES2022 already present: {len(existing)} files in {FES2022_DIR}")

In [ ]:
def predict_fes2022(lon, lat, timestamps, model_dir):
    """
    Predict tide height at (lon, lat) for a list of UTC timestamps using FES2022.
    Returns numpy array of tide heights in meters.
    """
    epoch = pd.Timestamp("2000-01-01", tz="UTC")
    delta_time = np.array(
        [(t - epoch).total_seconds() for t in timestamps], dtype=float
    )
    tide = pyTMD.compute.tide_elevations(
        np.atleast_1d(float(lon)),
        np.atleast_1d(float(lat)),
        delta_time,
        DIRECTORY=str(model_dir),
        MODEL="FES2022",
        EPOCH=(2000, 1, 1, 0, 0, 0),
        TYPE="time series",
        TIME="UTC",
        FILL_VALUE=np.nan,
    )
    return np.asarray(tide).flatten()

# Smoke test on first point
pt0 = points.iloc[0]
pt0_rows = df[df["point_id"] == pt0["id"]]
pt0_pred = predict_fes2022(pt0["lon"], pt0["lat"], pt0_rows["timestamp"].tolist(), FES2022_DIR)
print(f"FES2022 predictions for {pt0['id']} ({len(pt0_pred)} values): {pt0_pred[:5]}")

In [ ]:
assert SHOA_CSV.exists(), (
    f"{SHOA_CSV} not found. Export Bahía Orange tide predictions from "
    "http://www.shoa.cl/php/mareas.php (2023-07-01 → present) and save "
    "with columns: datetime (UTC ISO 8601), tide_m (meters)."
)

shoa_raw = pd.read_csv(SHOA_CSV, parse_dates=["datetime"])
shoa_raw["datetime"] = pd.to_datetime(shoa_raw["datetime"], utc=True)
shoa_ts = shoa_raw.set_index("datetime").sort_index()

def predict_shoa(point_id, timestamps):
    """
    Interpolate Bahía Orange patron tide at given timestamps, applying
    the published secondary port time offset for P07 and P12.
    All other points use the raw Bahía Orange value (offset = 0).
    Returns numpy array of tide heights in meters.
    """
    offset_min = SHOA_OFFSETS.get(point_id, 0)
    adjusted = [t - pd.Timedelta(minutes=offset_min) for t in timestamps]
    shoa_epoch = shoa_ts.index.map(lambda x: x.timestamp()).values
    shoa_vals = shoa_ts["tide_m"].values
    return np.array([
        np.interp(t.timestamp(), shoa_epoch, shoa_vals,
                  left=np.nan, right=np.nan)
        for t in adjusted
    ])

# Smoke test
pt0_shoa = predict_shoa(pt0["id"], pt0_rows["timestamp"].tolist())
print(f"SHOA predictions for {pt0['id']}: {pt0_shoa[:5]}")

In [ ]:
records = []
for pt_id, group in df.groupby("point_id"):
    pt_info = points[points["id"] == pt_id].iloc[0]
    timestamps = group["timestamp"].tolist()

    fes_pred = predict_fes2022(pt_info["lon"], pt_info["lat"], timestamps, FES2022_DIR)
    shoa_pred = predict_shoa(pt_id, timestamps)

    for i, (_, row) in enumerate(group.iterrows()):
        records.append({
            "point_id": pt_id,
            "timestamp": row["timestamp"],
            "wse": row["wse"],
            "wse_qual": row["wse_qual"],
            "tide_fes2022": fes_pred[i],
            "tide_shoa": shoa_pred[i],
            "residual_fes": row["wse"] - fes_pred[i],
            "residual_shoa": row["wse"] - shoa_pred[i],
        })

results = pd.DataFrame(records)
print(results.groupby("point_id")[["residual_fes", "residual_shoa"]].describe().round(3))